In [1]:
import os
import sys
import pandas as pd
sys.path.append('..')

from src.vectorstore import csv_loader, build_vectorstore
from src.rag_pipeline import load_llm, semantic_retriever, initialize_rag_chain, format_docs_to_df
from src.hybrid import preprocess_query, bm25_retriever, hybrid_retriever
from src.prompts import prompt, prompt_no_instruct, prompt_concise
from IPython.display import Markdown, display

# Set max width for results dataframe
pd.set_option('display.max_colwidth', 200)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Define model

In [3]:
llm = load_llm()

# RAG Semantic

## Load vector store

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

corpus_path = '../data/processed/preprocessed_corpus.csv'
vector_path = "../data/processed/vector_store"
docs = csv_loader(corpus_path)

if not os.path.exists(vector_path):
    build_vectorstore(docs, vector_path, embeddings)
    print(f"Saved vector store to {vector_path}")
    
vectorstore = FAISS.load_local(
    vector_path, embeddings, allow_dangerous_deserialization=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieval

In [5]:
vector_retriever = semantic_retriever(vectorstore)
query = "what is the best soap"
format_docs_to_df(vector_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B071VZQVLP,Savage Soaps Vajingo Organic Handcrafted Feminine Hygiene V Soap Bar,2.0,Soap did not perform as expected and did not last as long as soaps used in relation to size. Scent was mild and not aromatic as I was hoping it to be.
1,B01IJA11F0,"Dawn Professional Dish Soap 5 Gallon, Clean Scent (70681)",5.0,I really like Dawn dish soap every effective<br />I do housecleaning<br />Personal it does a super job
2,B0716PQVP2,Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool,5.0,Must have with bars of soap ! You will love !
3,B08DV37PZV,"Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack",5.0,"That ""blue"" dish soap is more difficult to rinse off. I like Palmolive because it cleans well and rinses off easily."
4,B001HDZT7I,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,Although this had decent reviews when I researched I learned very quickly that once these are wet they are not usable. The sheet size maybe can cover hand washing but they clump up and there isn't...


## Pipeline

In [6]:
def text(rag_chain, query):
    display(Markdown(str(rag_chain.invoke(query))))

In [7]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt)
text(rag_chain, query)

Based on the reviews provided, it's difficult to determine a single best soap as the reviews are primarily based on different types of soaps (feminine hygiene soap, dish soap, hand soap, and exfoliating soap saver). 

However, if we consider dish soap as the primary option, both Dawn Professional Dish Soap (B01IJA11F0) and Palmolive Ultra Original Dish Liquid (B08DV37PZV) received 5.0/5.0 ratings from customers, with praise for their effectiveness and ease of rinsing off. Between the two, the customer review for Palmolive mentions that it "cleans well and rinses off easily," which could make it a better choice for those who value this feature. 

Therefore, based on the provided reviews, Palmolive Ultra Original Dish Liquid (B08DV37PZV) could be considered a top option for dish soap.

In [8]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt_no_instruct)
text(rag_chain, query)

Based on the customer reviews provided, I can give you an analysis of the different soaps mentioned.

For dish soap, two top-rated options stand out:

1. **Dawn Professional Dish Soap 5 Gallon, Clean Scent (ASIN: B01IJA11F0)**: This dish soap has a perfect 5.0 rating and is effective for housecleaning. It's also super easy to use and clean.
2. **Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack (ASIN: B08DV37PZV)**: This dish soap also has a 5.0 rating and is praised for its ability to clean well and rinse off easily.

However, when it comes to the best overall soap option, it's tricky to determine since the customer reviews are specifically for different types of soap (dish soap vs. handcrafted feminine hygiene soap bar).

If you're looking for a dish soap, the two mentioned above are excellent options. If you're looking for a handcrafted feminine hygiene soap bar, I would recommend checking the reviews for other products in that category, as the Savage Soaps Vajingo Organic Handcrafted Feminine Hygiene V Soap Bar (ASIN: B071VZQVLP) received a relatively lower rating of 2.0 out of 5.0.

For a soap saver pouch, the Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool (ASIN: B0716PQVP2) is a highly rated option with a 5.0 rating.

As for a travel-friendly hand soap option, the Travelon Hand Soap Toiletry Sheets, 50-Count (ASIN: B001HDZT7I) received a 3.0 out of 5.0 rating, which may not be the best option as it has some usability concerns.

Ultimately, the best soap for you will depend on your specific needs and preferences. If you have any further questions or would like more recommendations, feel free to ask!

In [9]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt_concise)
text(rag_chain, query)

Based on the reviews, it's difficult to pinpoint the "best soap" as the question is subjective and depends on specific needs and preferences. However, if I had to choose a top-rated soap for a specific task, here are some recommendations:

1. For handwashing, Palmolive Ultra Original Dish Liquid (ASIN: B08DV37PZV) is a top choice, as it cleans well and rinses off easily.
2. For dishwashing, Dawn Professional Dish Soap (ASIN: B01IJA11F0) is a top pick, as it's effective and well-suited for housecleaning tasks.
3. For a travel option, the reviews didn't directly recommend a soap, but if I had to choose, I'd suggest a bar soap that's known for being easy to travel with, such as Savage Soaps Vajingo Organic Handcrafted Feminine Hygiene V Soap Bar (ASIN: B071VZQVLP), despite its mixed reviews.

Keep in mind that these are just based on the reviews provided and may not reflect the best soap for your specific needs.

# Hybrid RAG: Semantic Search + BM25

### BM25 Retriever

In [10]:
keyword_retriever = bm25_retriever(docs)

# preprocess query so it's consistent with corpus preprocessing
query = preprocess_query(query)

# Retriever invoke returns the top_k docs
format_docs_to_df(keyword_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B01HLGEPKQ,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap (Pack of 8),5.0,It's good soap. What more can I say?
1,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",4.0,A single sheet is good for a pair of underwear. The soap dissolves quickly but the water does not feel too soapy so not sure about the quality of cleaning. Best is to let the clothes sit in the so...
2,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,"If you MUST save on space, consider these laundry soap sheets. I will be honest, though... hotel bar soap and baby wash made my clothes smell (and therefor feel) cleaner to me when doing laundry ..."
3,B07THJSBH1,Soap Saver Pouch | Exfoliating Sponge Soap Pocket Body Exfoliation for Bath or Shower | Body Scrubber for Bar Soap or Soap Bits | 2 Soap Bag + 2 Piece Soap Saver Pads + 2 Hooks,5.0,"These are great and well made. The material is sisal so it’s a little scratchy, but considering you’re using it to get clean it’s a plus. Similar to a loofa. The bags hold up in the washer too; I ..."
4,B00OPC61V6,"Dial Mountain Fresh Antibacterial Deodorant Soap, 4 oz, 6 Count",5.0,"I use Dial soap because it does the best job of fighting odor, particularly during the summer when it's so hot and humid in the South. Leaves me feeling clean and fresh, rinses well, nice fragran..."


## Ensemble BM25 + Semantic Retriever

In [11]:
# Invoke to get combined results
ensemble_retriever = hybrid_retriever(keyword_retriever, vector_retriever)
format_docs_to_df(ensemble_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B01HLGEPKQ,Dial Corp. 04303 Fels-Naptha Laundry Bar Soap (Pack of 8),5.0,It's good soap. What more can I say?
1,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",4.0,A single sheet is good for a pair of underwear. The soap dissolves quickly but the water does not feel too soapy so not sure about the quality of cleaning. Best is to let the clothes sit in the so...
2,B071VZQVLP,Savage Soaps Vajingo Organic Handcrafted Feminine Hygiene V Soap Bar,2.0,Soap did not perform as expected and did not last as long as soaps used in relation to size. Scent was mild and not aromatic as I was hoping it to be.
3,B01IJA11F0,"Dawn Professional Dish Soap 5 Gallon, Clean Scent (70681)",5.0,I really like Dawn dish soap every effective<br />I do housecleaning<br />Personal it does a super job
4,B001CGOPZM,"Travelon Hand Soap Toiletry Sheets, 50-Count",5.0,nice product
5,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,"If you MUST save on space, consider these laundry soap sheets. I will be honest, though... hotel bar soap and baby wash made my clothes smell (and therefor feel) cleaner to me when doing laundry ..."
6,B07THJSBH1,Soap Saver Pouch | Exfoliating Sponge Soap Pocket Body Exfoliation for Bath or Shower | Body Scrubber for Bar Soap or Soap Bits | 2 Soap Bag + 2 Piece Soap Saver Pads + 2 Hooks,5.0,"These are great and well made. The material is sisal so it’s a little scratchy, but considering you’re using it to get clean it’s a plus. Similar to a loofa. The bags hold up in the washer too; I ..."
7,B00OPC61V6,"Dial Mountain Fresh Antibacterial Deodorant Soap, 4 oz, 6 Count",5.0,"I use Dial soap because it does the best job of fighting odor, particularly during the summer when it's so hot and humid in the South. Leaves me feeling clean and fresh, rinses well, nice fragran..."


In [12]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt)
text(rag_chain, query)

Based on the reviews, it seems that several soaps are considered "good soap" by customers. However, to provide a more specific answer, I would recommend the following products:

1. **Fels-Naptha Laundry Bar Soap** (Product ASIN: B01HLGEPKQ) - This product has a 5-star rating and a simple review stating that it's a "good soap." It's available in a pack of 8.

2. **Dial Mountain Fresh Antibacterial Deodorant Soap** (Product ASIN: B00OPC61V6) - This product has a 5-star rating and is praised for its ability to fight odor, leaving the user feeling clean and fresh.

3. **Dawn Professional Dish Soap** (Product ASIN: B01IJA11F0) - Although this product is a dish soap, it's mentioned in a review as being "every effective" and a super job at cleaning. While it's not explicitly stated as a personal soap, its effectiveness might make it a good alternative.

Please note that these recommendations are based solely on the provided reviews and do not include any information about the actual quality or characteristics of the soaps.

In [13]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt_no_instruct)
text(rag_chain, query)

It seems like you're looking for a good soap. Based on the reviews provided, here are some options:

1. **Dial Corp. 04303 Fels-Naptha Laundry Bar Soap (Pack of 8)**: This soap has a 5.0/5.0 rating and a simple, straightforward review: "It's good soap. What more can I say?" It's a laundry bar soap, but it's a good option if you're looking for something basic and effective.

2. **Dial Mountain Fresh Antibacterial Deodorant Soap**: This soap also has a 5.0/5.0 rating and is praised for its ability to fight odor and leave the skin feeling clean and fresh. It's available in a 4 oz, 6-count pack.

Both of these options have high ratings and positive reviews, so you can't go wrong with either choice.

If you're looking for something more specific (e.g., hand soap, dish soap), I'd be happy to help with that as well!

In [14]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt_concise)
text(rag_chain, query)

Based on the reviews, Fels-Naptha Laundry Bar Soap (ASIN: B01HLGEPKQ) is described as "good soap" with a 5-star rating.

# Qualitative Evaluation for Hybrid RAG Workflow

In [15]:
# query list from Milestone 1
queries = ["Wet wipes",
           "Bar Soap",
           "Small hair dryer",
           "The best air humidifer with essential oil",
           "mineral sunscreen for babies",
           "hair spray that last more than 6 hours",
           "Best Vitamins or supplements to take for pregnant women",
           "equipments for stretching at home",
           "sunrise lamp that will help me to wake up in the morning",
           "Something to relieve my back pain"]

In [16]:
# use original prompt
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt)
for q in queries:
    display(Markdown(f"##QUERY: {q}\n"))
    text(rag_chain, q)
    display(Markdown("### Sources"))
    retrieved_docs = ensemble_retriever.invoke(q)
    df = format_docs_to_df(retrieved_docs)
    display(df)

##QUERY: Wet wipes


Based on the provided Amazon product reviews, here are some key points about wet wipes:

1. Thick and effective: 
   - The Rolhei 75% Ethanol Wet Wipe (B08HVXPVWV) review mentions that the wipes are thick.
   - The Kandoo Flushable Wipes (B003FGE1TC and B006KYCY5Y) reviews mention that the wipes are nice and thick, comparable to Cottonelle and Huggies.

2. Multi-purpose:
   - The Center Pull Spunlace Dry Wipes (B08RRFL7PL) review mentions that the towels can be used as dry wipes or refills for wet wipes, and can be used in various places such as the bathroom, garage, laundry room, and more.
   - The Kandoo Flushable Wipes (B003FGE1TC and B006KYCY5Y) review mentions using the wipes for potty training and other purposes.

3. Suitable for different skin types:
   - The Kandoo Flushable Wipes (B003FGE1TC and B006KYCY5Y) are mentioned as being unscented and hypoallergenic, making them suitable for sensitive skin.

4. Travel-friendly:
   - The Systane Lid Wipes (B00HHXFQSY) reviews mention that the wipes are great for travel and cleaning eyes.

Note that the products mentioned above offer a mix of dry and wet wipes, depending on the customer's use case and preferences.

### Sources

,Product ASIN,Title,Rating,Review
0,B08RRFL7PL,"Center Pull Spunlace Dry Wipes, 6 Rolls, Multi-Use Hand Towels, Refill For Bucket Wipes To Create Own Cleaning Wipes, Ships Dry Or Use In Center Pull Dispenser Hand Towels",5.0,"I have two bucket dispensers for these towels, and I use the refills in them and in so many other places. I have never made wipes, but just use the towels/wipes dry. Having the center pull is so e..."
1,B003FGE1TC,"Flushable Wipes for Baby and Kids by Kandoo, Unscented for Sensitive Skin, Hypoallergenic Potty Training Wet Cleansing Cloths, 42 Count, Pack of 12",5.0,"I admit, I bought these because I couldn't find my normal disposable wipes anywhere. I didn't think I'd be impressed, but in 2020 - you make do. Well, I am so pleasantly surprised with these. Not ..."
2,B08HVXPVWV,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (200 Wipes),5.0,that the wipes are thick and not thin.
3,B01IAIONNI,20CT Wet Ones Wipes - Pack of 8,5.0,we love the wet ones.
4,B006KYCY5Y,"Flushable Wipes for Baby and Kids by Kandoo, Unscented for Sensitive Skin, Hypoallergenic Potty Training Wet Cleansing Cloths, 42 Count, Pack of 12",5.0,"These wipes are nice & thick, they smell good also. An the price is good. I'm going to buy more because I always use wipes. An these are close to Cottonelle & Huggies wipes. What I really like is..."
5,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",4.0,I’m not impressed.
6,B08QGXZRLB,"Lens Wipes - 400 Eye Glasses Cleaning Wipe, Our Eyeglass Cleaner, Screen Wipes, Eyeglass Wipes, Lens Cleaning Wipes - Glasses Wipes pre-moistened",5.0,"I use them everyday to wipe my daughters iPad, sunglasses and my regular glasses too. They are a little dry but they still work great!"
7,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",5.0,Very good product
8,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",5.0,Great for travel and to clean eyes.
9,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",4.0,The wipes are large and easy to use.


##QUERY: Bar Soap


...

Based on the reviews above, it appears that the following supplements may be beneficial for pregnant women:

1. La Dra. Miller Vegetal Compound (ASIN: B07DRMR6VV): This product has been reviewed by a user who claims that it helped them conceive after drinking three bottles back-to-back.
2. Baby 'n' Tot Vitamin D Drops (ASIN: B00TJ2RJBI): While not specifically marketed for pregnant women, this product is designed to support immune function and has been reviewed by a user as a legitimate and safe option.

Please note that it's essential to consult with a healthcare professional before taking any supplements during pregnancy. The claims made in the reviews above are anecdotal, and it's crucial to prioritize individual health and consult with a medical professional for personalized advice.

### Sources

,Product ASIN,Title,Rating,Review
0,B015X2DNIU,"Krill Oil Brain & Vision Formula with Lutein. Supports Cognitive Function, Improved Memory, Protects Eyes & Boosts Vision",5.0,"First off, I bought this product for my grandmother who was diagnosed with dementia almost a year ago. I had seen it as a lightning deal and decided to try it for her in addition to her medication..."
1,B08HHTN7XG,"Wellmo Natural Vitamin D3 Soft Gels in Fast-Absorbing Coconut Oil for Men, Women, and Children - All-Natural, Vitamin D3 for Bone, Skin, and Immune Health - 60 Soft Gels (60 Day Supply)",5.0,"Even being in the sun a lot, my blood work always shows me a little low on Vit D.<br />I like these pills. Nice and small to swallow. Helps boost my Vit D levels so there's no concerns.<br />I l..."
2,B08VRVQ9P9,"Nature's Clear Multivitamin for Women and Men With Vitamins A, C, D, E, Vitamin B12, Zinc, Biotin, Calcium & Magnesium, Daily Immune Health Support, 60 Capsules",3.0,I needed another multivitamin and found this one. I liked that it had Chlorophyll and CQ10. As well as a good amount of copper. Those are things I needed to help me get used to the high altitudes ...
3,B017V3058K,Numedica- Omega 780 - 120 Softgels (Premium Packaging),5.0,This is necessary when I take glucosatrin. This works together and adds vitamin A. Works good.
4,B00TJ2RJBI,"Baby 'n' Tot Vitamin D Drops, Enjoy Over 530 Servings of High Quality, Safe and Natural Liquid Vitamin D3 Specially Designed for Breast & Mixed Fed Babies, Supercharge Your Child's Immune System, ...",4.0,Seems to be legit. I get a little nervous buying from 3rd parties due to possible knock offs.
5,B0006MQ5OM,"Hyland's - Calc. Fluor 6x, 500 Tablets",5.0,"I was skeptical kinda, I feel it really works actually. I think this and evening primrose an magnesium citrate help me get pregnant. I had extreme infertility problems took 5 years to get pregnant..."
6,B07DRMR6VV,La Dra. Miller 16 oz. Vegetal Compound,5.0,I bought 3.i am 40.. I want to make sure i was clean and ready. At the same time scared. I had took plan B before.. I drink 3 back to back to be sure and Now i am having a baby boy. It really work...
7,B0827WT513,"PHIMIITA Pill Organizer Large Compartments Pill Case 4 Times a Day Medicine Organizer to Hold Vitamins, Supplements,Medication and Fish Oil",5.0,Works perfectly for someone who takes many pills
8,B09Q9CYZWF,"Trace Minerals | Liquid Gut Health | Support for Gut Lining, Normal Digestion, Immunity, Metabolism, Detoxification | 10 mg Humic & Fulvic Acid | Unflavored | 32 fl oz",5.0,"I can't directly say how good this works since am nursing, I didn’t realize it's not for recommend for anyone nursing or pregnant. But I have been giving it to my son who is a picky eater a..."
9,B08B1HRVDS,"Organic Forskolin Extract Capsules 5000MG with Turmeric Curcumin, Arjuna, Garcinia Cambogia, Green Tea | Maximum Strength India Supplements | Support Nourishing Diet Coleus Forskohlii, Energy Booster",5.0,These capsules work very well when taken at night and in the morning on an empty stomach.


##QUERY: equipments for stretching at home


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/novita/v3/openai/chat/completions' (Request ID: Root=1-69e88261-1e24f6d02b7acb7846c2ce35;06465643-67fb-41cf-89ff-45372d7be741)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

# Experiment with the LLM

In [19]:
# Try another additional LLM model
llm2 = load_llm(repo_id="Qwen/Qwen3-8B", provider="fireworks-ai")

In [20]:
rag_chain2 = initialize_rag_chain(ensemble_retriever, llm2, prompt)
for q in queries:
    display(Markdown(f"##QUERY: {q}\n"))
    text(rag_chain2, q)
    display(Markdown("### Sources"))
    retrieved_docs = ensemble_retriever.invoke(q)
    df = format_docs_to_df(retrieved_docs)
    display(df)

##QUERY: Wet wipes


<think>
Okay, the user asked about "wet wipes" based on the provided reviews. Let me go through each product ASIN to find relevant information.

First, looking at ASIN B08RRFL7PL: The review mentions using the towels dry, but they are referred to as "dry wipes" or "hand towels." Not sure if they're wet wipes here. The product title says "Center Pull Spunlace Dry Wipes," so probably not wet.

Next, B003FGE1TC: These are flushable wipes for baby and kids, unscented. The review says they do the job well and are soft. They are wet wipes since they're for cleansing. The user mentioned they're "wet cleansing cloths."

ASIN B08HVXPVWV: Title is "Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (200 Wipes)" with a 5.0 rating. Review says the wipes are thick and not thin. So this is definitely wet wipes, alcohol-based.

ASIN B01IAIONNI: "20CT Wet Ones Wipes - Pack of 8" with a 5.0 rating. Review says "we love the wet ones," which refers to the product name. So these are wet wipes.

ASIN B006KYCY5Y: Same product as B003FGE1TC, flushable wipes for baby, 42 count. Review mentions they're nice and thick, no white fuzz. Also wet wipes.

ASIN B00HHXFQSY: Systane Lid Wipes, eyelid cleansing. The reviews mention they are sterile, used for cleaning eyes. The product is for eyelid, but they are wet wipes. Some reviews rate 4.0 and others 5.0. One review says "a little dry but still work great," so maybe they are pre-moistened but not super wet.

ASIN B08QGXZRLB: Lens Wipes for glasses, screen cleaning. The review mentions they are pre-moistened. The title says "pre-moistened," so these are wet wipes for cleaning lenses.

Now, compiling all these. The user asked about wet wipes, so I need to list the products that are wet wipes. The answer should mention each relevant ASIN with their details. Need to check if all the mentioned products are indeed wet

### Sources

,Product ASIN,Title,Rating,Review
0,B08RRFL7PL,"Center Pull Spunlace Dry Wipes, 6 Rolls, Multi-Use Hand Towels, Refill For Bucket Wipes To Create Own Cleaning Wipes, Ships Dry Or Use In Center Pull Dispenser Hand Towels",5.0,"I have two bucket dispensers for these towels, and I use the refills in them and in so many other places. I have never made wipes, but just use the towels/wipes dry. Having the center pull is so e..."
1,B003FGE1TC,"Flushable Wipes for Baby and Kids by Kandoo, Unscented for Sensitive Skin, Hypoallergenic Potty Training Wet Cleansing Cloths, 42 Count, Pack of 12",5.0,"I admit, I bought these because I couldn't find my normal disposable wipes anywhere. I didn't think I'd be impressed, but in 2020 - you make do. Well, I am so pleasantly surprised with these. Not ..."
2,B08HVXPVWV,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (200 Wipes),5.0,that the wipes are thick and not thin.
3,B01IAIONNI,20CT Wet Ones Wipes - Pack of 8,5.0,we love the wet ones.
4,B006KYCY5Y,"Flushable Wipes for Baby and Kids by Kandoo, Unscented for Sensitive Skin, Hypoallergenic Potty Training Wet Cleansing Cloths, 42 Count, Pack of 12",5.0,"These wipes are nice & thick, they smell good also. An the price is good. I'm going to buy more because I always use wipes. An these are close to Cottonelle & Huggies wipes. What I really like is..."
5,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",4.0,I’m not impressed.
6,B08QGXZRLB,"Lens Wipes - 400 Eye Glasses Cleaning Wipe, Our Eyeglass Cleaner, Screen Wipes, Eyeglass Wipes, Lens Cleaning Wipes - Glasses Wipes pre-moistened",5.0,"I use them everyday to wipe my daughters iPad, sunglasses and my regular glasses too. They are a little dry but they still work great!"
7,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",5.0,Very good product
8,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",5.0,Great for travel and to clean eyes.
9,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",4.0,The wipes are large and easy to use.


##QUERY: Bar Soap


...

,Product ASIN,Title,Rating,Review
0,B00BYG3OWK,Naturebright L6060 Per2 Led Daylight Lamp,4.0,"I tried the Phillips previously (see review there) but just couldn't get it to work. The NatureBright is easy to figure out, the controls make a lot more sense and you can easily see that it's se..."
1,B000F95A4W,"Verilux Original Natural Spectrum Deluxe Floor Lamp, Ivory",5.0,"I use this lamp at my bedside either for reading books or with my Kindle Fire for reading and apps. Before getting this lamp, I was content with using my overhead lighting to read in my room. Si..."
2,B00H86M7DW,6 Himalayan Salt Lamps with Wood Base (up to 4 lbs),3.0,I like the lamps but the bases don't hold in the lamps well & the bulbs blow in days. I would buy again but be prepared to put them together and purchase extra bulbs. I make sure to turn them off...
3,B00BYG3OWK,Naturebright L6060 Per2 Led Daylight Lamp,3.0,"This functions well as daylight alarm. It's best feature is that the LED's arc out over the bed and shine down, as opposed to other models which shine straight out horizontally, which means you ha..."
4,B0041HU4YK,D,4.0,I got this clock for my son who had trouble getting up at 5:30AM in the darkness. He liked it but in his sleep would set the 10 minute snooze and would wake up puzzled that he over slept. It is a ...
5,B0041HU4YK,D,5.0,"I have such a hard time waking up in the morning, even when I get 8 hours of sleep. There are times when I sleep through the loud buzzing of my alarm clock. When the days started getting shorter a..."
6,B003DS209K,"5-hour ENERGY Shot, Regular Strength Orange, 1.93 oz., 24 pack",5.0,I got what I needed to wake up and get out to my commute while it's still dark. I am the poster child for &#34;not a morning person.&#34; This makes it happen.
7,B07H4J2CVH,"Sleep Mask for Women and Men,3D Contoured Eye Mask for Sleeping Mask Eye Cover,Lightweight Sleep Masks Blindfolds Eye Shade for Kids Girls Travel,2 Pack Black",5.0,"Very comfortable, which surprised me! I love that it doesn't press against your eyes. I also suffer from extremely dry eyes, especially in the morning when I wake up, due to multiple fans running ..."
8,B003TPXYF6,Malem Wireless Bedwetting Alarm System,5.0,My son is almost 9yrs old. He had been wet EVERY single night since birth. He would often wet through pull-ups. It was really starting to effect his self-esteem. Sleep-overs and scout trips wer...


##QUERY: Something to relieve my back pain


<think>
Okay, the user is looking for something to relieve their back pain. Let me go through the product reviews provided and see which ones are relevant.

First, looking at the ASIN B00MCWFNOM, it's a Battle Balm Stick. The review says it's very greasy and doesn't relieve pain, so probably not a good choice. 

Next, B08NVFSSH8 is a Back Massage Stretcher. The review mentions it's a bit stiff for sore muscles and needs a soft pad. Maybe not the best if they need something flexible. 

B08CVPZCP9 is another back stretcher. The review says it helps with lower back stretching, not a cure but provides relief. That's positive. 

B019O0QNT8 is a seat cushion for sciatica and hip pain. The review says it works well but mentions orientation. Could be useful if they sit a lot. 

B074QLYVF9 is a back stretcher for sciatica. The user found it uncomfortable but their wife likes it. Maybe a good option if they can adjust. 

B08Z2SDRL2 is a neck stretcher, but the user asked for back pain. Not directly relevant unless they have neck issues too. 

B01MR6QQBB is a shoulder wrap. The review says it's good for muscle pain, but the user's issue is back pain. Maybe not. 

B0787TBYP9 is a posture corrector brace. The review mentions relief from back pain. That's a good candidate. 

B015MUK59S is a patch for pain relief, but the review says it's for other areas like shoulder and arthritis. Not sure about back pain specifically. 

So the top options seem to be B08CVPZCP9 (PGFUN Back Stretcher), B074QLYVF9 (Aptoco Back Stretcher), and B0787TBYP9 (Posture Corrector). The user might need something that's comfortable and provides relief. The posture corrector has a 5-star rating and directly mentions back pain relief. The back stretchers have good ratings but some mention stiffness. The seat cushion is also a 5-star but for sciatica. 

I should highlight the posture corrector and the back stretchers, noting their ratings and specific benefits. Also, mention the seat cushion as an option.

### Sources

,Product ASIN,Title,Rating,Review
0,B00MCWFNOM,"Battle Balm Stick - Demon Strength - Natural Herbal Pain Relief for Arthritis, Sciatica, Back, Neck, Leg, Shoulder - Also Relieves Workout Soreness (0.75 oz.)",2.0,Very greasy and although I feel heat it doesn't relieve the pain
1,B08NVFSSH8,"Genericb Back Massage Stretcher Arch Magic Message Stretcher Back Stretcher Lumbar Support Device, Lower and Upper Back Pain Relief Relax Mate Spine Pain Relief Chiropractic",4.0,A little to stiff for sore back muscles. need to use a soft pad with it
2,B08CVPZCP9,"PGFUN Back Stretcher, Multi-Level Back Stretching Device for Upper and Lower Muscle Pain Relief, Back Massager Spinal Lumbar Support for Herniated Disc, Sciatica, Scoliosis, Spine Deck",4.0,This helps stretching my lower back. Not a cure but any relief is better than nothing.
3,B019O0QNT8,"CareU(™) Coccyx Orthopedic Portable Comfort Memory Foam Seat Cushion with Non Slip Cover- Helps Relieve Kyphotic, Hip and Sciatica Pain (Standard Size 17.7x15 Inch)",5.0,Works well. Just make sure the front is the front and the back is the back as intended.
4,B074QLYVF9,"Back Massager, Aptoco Back Stretcher for Sciatica Relief, Herniated Disc/Spinal Stenosis-Back Massage Stretcher, Back Stretching Device",4.0,A little uncomfortable for me (not very flexable) but my wife loves it and uses it multiple times a day.
5,B08Z2SDRL2,"Neck Stretcher Cervical Neck Traction Device Over Door for Home Use,Portable Neck Traction for Neck Pain Relief, Physical Therapy AIDS for Neck Spine Decompressor (Black)",5.0,First time user. It took me a while a find out how to assemble them. After some adjustments I find a way to use it. And it does helps relieve my neck pain.
6,B01MR6QQBB,Shoulder Wrap Gel Ice Hot Cold Pack for Shoulder Injury Pain Relief Therapy Rotator Cuff Rheumatoid Arthritis Treatment Osteoarthritis Bursitis Tendinitis AC Joint Sports Injuries,5.0,"I use it alot, great product, helps relieve my muscle pain."
7,B0787TBYP9,"ZSZBACE Posture Corrector Back Brace for Men and Women- Relieve Back Pain, Align Spain, Correct Kyphosis (XXL)",5.0,It help with my back pain
8,B015MUK59S,Korean Red Ginseng Patch Powerstrip Energy Pain Relief - 20 Patches,5.0,These patches really do relieve the pain. They can be cut to whatever size you need.They stick extremely well.
